In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {"travel_server": {"transport": "streamable_http", "url": "https://mcp.kiwi.com"}}
)

tools = await client.get_tools()

In [3]:
for idx, t in enumerate(tools, start=1):
    print(f"\n- - - - - TOOL {idx} - - - - -")
    print(t.name)
    print(t.description)


- - - - - TOOL 1 - - - - -
search-flight
# Search for flights

Searches Kiwi.com for available flights between two locations for the given dates and passengers. City or airport names are resolved automatically, so call this whenever the user wants to search for flights — whether they gave IATA codes or just place names.

## Result shape

Returns `{ query, currency, passengers, resultsCount, itineraries, searchTimeMs }`. Each item in `itineraries` has:
- `price` (number) and `priceFormatted` (e.g. "123 EUR")
- `totalDurationSeconds`
- `bookingUrl` — the link to book the flight
- `imageId` — destination city id for a hero photo (https://images.kiwi.com/photos/600x600/{imageId}.jpg)
- `baggage` — total included baggage across all travelers: `{ personalItem, cabinBag, checkedBag }` (counts)
- `outbound` (and `inbound` for return flights), each a leg with: `route` (list of airport codes including layovers, e.g. ["PRG","MAD","BCN"]), `departureTime` / `arrivalTime` (local ISO timestamps), `

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions.",
)

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(
                content="Get me a direct flight from San Francisco to Tokyo on March 31st"
            )
        ]
    },
    config,
)

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st', additional_kwargs={}, response_metadata={}, id='5081c6db-9318-4403-97a4-537d886b0878'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 789, 'prompt_tokens': 2693, 'total_tokens': 3482, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 704, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EZZWniiry37ExMTVczprACtnKOJ7d', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a14445-5205-7b50-ba8a-770d237bea9f-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'San Francisco', 'flyTo': 'Tokyo', 'departureDate': '31/03/2027

In [7]:
print(response["messages"][-1].content)

Here are direct-flight options from San Francisco (SFO) to Tokyo on March 31, 2027. All are one-way direct flights (no stops).

| Route | Outbound (local times) | Cabin | Duration | Price | Booking |
|---|---|---|---:|---:|---|
| SFO → NRT | 12:20 31 Mar 2027 → 15:25 01 Apr 2027 | Economy | 11h 5m | 508 EUR | https://kiwi.com/u/ptynwb |
| SFO → NRT | 16:50 31 Mar 2027 → 20:00 01 Apr 2027 | Economy | 11h 10m | 546 EUR | https://kiwi.com/u/kjqfj9 |
| SFO → NRT | 12:20 31 Mar 2027 → 15:25 01 Apr 2027 | Economy | 11h 5m | 975 EUR | https://kiwi.com/u/gz9ft4 (NH7013) |
| SFO → NRT | 13:30 31 Mar 2027 → 16:30 01 Apr 2027 | Economy | 11h 0m | 1110 EUR | https://kiwi.com/u/f4373k (AA8410) |
| SFO → NRT | 11:55 31 Mar 2027 → 15:05 01 Apr 2027 | Economy | 11h 15m | 7339 EUR | https://kiwi.com/u/urwfcj (JL1) |

Notes:
- The cheapest option is 508 EUR (NRT) with All Nippon Airways (NH7) departing 12:20 from SFO and arriving 15:25 next day at Narita.
- Duration is outbound-only since this is a one-